# 사업부오더 일별집계

- **원본**: `서비스오더.pkl` (C:\DA)
- **기간**: 2023-01 ~ 2026-04 (`서비스오더(YY.MM월)` 형식 시트)
- **목적**: 사업부(서비스처리센터)별 일별 서비스오더 (마감일 기준) 건수 집계

## 1단계: 라이브러리 임포트

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("라이브러리 로드 완료")

라이브러리 로드 완료


## 2단계: 파일 경로 정의

In [2]:
PKL_CANDIDATES = [
    Path(r"C:\DA\서비스오더.pkl"),
    Path(r"C:\DA\23_26년_서비스오더.pkl"),
    Path(r"\\DocuONE\MyDrive\개인함\23_26년_서비스오더\서비스오더.pkl"),
]

PKL_PATH = next((p for p in PKL_CANDIDATES if p.exists()), None)
print(f"사용할 pkl: {PKL_PATH}")

사용할 pkl: \\DocuONE\MyDrive\개인함\23_26년_서비스오더\서비스오더.pkl


## 3단계: 데이터 로드

In [3]:
print(f"로드 중... ({PKL_PATH.name})")
df = pd.read_pickle(PKL_PATH)
df.columns = [str(c).strip() for c in df.columns]
print(f"로드 완료: {len(df):,}행")

로드 중... (서비스오더.pkl)


로드 완료: 1,833,764행


## 4단계: 서비스오더(YY.MM월) 시트만 필터

In [4]:
mask = df["시트명"].str.match(r"^서비스오더\(\d{2}\.\d{1,2}월\)$", na=False)
df = df[mask].copy()
print(f"필터 후: {len(df):,}행")
print(f"시트 목록: {sorted(df['시트명'].unique())}")

필터 후: 1,833,764행
시트 목록: ['서비스오더(23.01월)', '서비스오더(23.02월)', '서비스오더(23.03월)', '서비스오더(23.04월)', '서비스오더(23.05월)', '서비스오더(23.06월)', '서비스오더(23.07월)', '서비스오더(23.08월)', '서비스오더(23.09월)', '서비스오더(23.10월)', '서비스오더(23.11월)', '서비스오더(23.12월)', '서비스오더(24.01월)', '서비스오더(24.02월)', '서비스오더(24.03월)', '서비스오더(24.04월)', '서비스오더(24.05월)', '서비스오더(24.06월)', '서비스오더(24.07월)', '서비스오더(24.08월)', '서비스오더(24.09월)', '서비스오더(24.10월)', '서비스오더(24.11월)', '서비스오더(24.12월)', '서비스오더(25.01월)', '서비스오더(25.02월)', '서비스오더(25.03월)', '서비스오더(25.04월)', '서비스오더(25.05월)', '서비스오더(25.06월)', '서비스오더(25.07월)', '서비스오더(25.08월)', '서비스오더(25.09월)', '서비스오더(25.10월)', '서비스오더(25.11월)', '서비스오더(25.12월)', '서비스오더(26.01월)', '서비스오더(26.02월)', '서비스오더(26.03월)', '서비스오더(26.04월)']


## 5단계: H051 → H073 통합

In [5]:
df["서비스처리센터"] = df["서비스처리센터"].astype(str).str.strip().replace({"H051": "H073"})
print(f"사업부 분포:\n{df['서비스처리센터'].value_counts()}")

사업부 분포:
서비스처리센터
H074    409326
H073    405535
H075    359545
H072    339794
H071    319534
nan         25
H061         4
2            1
Name: count, dtype: int64


## 6단계: 소분류 필터 (6개 핵심 소분류 전체)

In [6]:
target_categories = ["전입신청", "전입변경", "전출신청", "전출변경", "연소기 연결/교체", "연소기철거"]
df = df[df["소분류"].isin(target_categories)].copy()
print(f"소분류 필터 후 (전체 오더): {len(df):,}행")
print(df["소분류"].value_counts())

소분류 필터 후 (전체 오더): 1,123,695행
소분류
전출신청         480219
전입신청         472260
전입변경          54533
전출변경          52168
연소기 연결/교체     48022
연소기철거         16493
Name: count, dtype: int64


## 7단계: 신규아파트 매칭 (apartments.txt × '전입신청/변경' 오더 기준 월별 15건 이상)

In [7]:
# 7단계: 데이터 정규화 및 관할 후보군 맵 로드
import re
from pathlib import Path

# 주소 정규화
df["주소_norm"] = df["주소"].astype(str).str.split("(").str[0].str.strip()

# 년도월 생성 (시트명 기반)
def ym_from_sheet(s):
    m = re.search(r"\((\d{2})\.(\d{1,2})월\)", str(s))
    return f"20{m.group(1)}-{int(m.group(2)):02d}" if m else ""
df["년도월"] = df["시트명"].map(ym_from_sheet)

# 통합본 정정 엑셀에서 하이브리드 관할 맵 및 제외량 로드
xlsx_path = Path("C:/DA/종합오더집계_통합본_정정.xlsx")
print(f"통합본 엑셀 로드 중: {xlsx_path.name}")

df_dong = pd.read_excel(xlsx_path, sheet_name='행정동_월별')
df_dong['서비스처리센터'] = df_dong['서비스처리센터'].ffill()
df_dong['행정동'] = df_dong['행정동'].ffill()

years = df_dong.columns[3:]
months = df_dong.iloc[0, 3:].values
ym_cols = []
valid_col_indices = []
current_year = ""
for idx, (y, m) in enumerate(zip(years, months)):
    m_str = str(m).strip()
    if m_str in ['합계', '총합', '소계', 'nan'] or pd.isna(m): continue
    if not str(y).startswith("Unnamed") and pd.notna(y):
        current_year = str(y).strip()
    try:
        m_int = int(float(m_str))
        ym_cols.append(f"{current_year}-{m_int:02d}")
        valid_col_indices.append(idx + 3)
    except ValueError:
        pass

# 1. 하이브리드 관할 후보군 맵 구축 (행정동_월별 시트의 '전체오더' 구분 기준)
ym_dong_candidates = {}
invalid_keywords = ['소계', '합계', '전체', '총합', 'nan', '']
for idx, row in df_dong.iterrows():
    dong_val = str(row['행정동']).strip()
    bu_val = str(row['서비스처리센터']).strip()
    gubun_val = str(row['구분']).strip()
    if dong_val in invalid_keywords or bu_val in invalid_keywords or gubun_val != "전체오더":
        continue
    for col_ym, col_idx in zip(ym_cols, valid_col_indices):
        val = row.iloc[col_idx]
        if pd.notna(val) and val > 0:
            key = (col_ym, dong_val)
            if key not in ym_dong_candidates:
                ym_dong_candidates[key] = {}
            ym_dong_candidates[key][bu_val] = int(val)

# 2. 신규아파트오더 제외 목표치 구축 (행정동_월별 시트의 '신규아파트오더' 구분 기준)
target_exclusions = {}
for idx, row in df_dong.iterrows():
    dong_val = str(row['행정동']).strip()
    bu_val = str(row['서비스처리센터']).strip()
    gubun_val = str(row['구분']).strip()
    if dong_val in invalid_keywords or bu_val in invalid_keywords or gubun_val != "신규아파트오더":
        continue
    for col_ym, col_idx in zip(ym_cols, valid_col_indices):
        val = row.iloc[col_idx]
        if pd.notna(val) and val > 0:
            target_exclusions[(bu_val, dong_val, col_ym)] = int(val)

print(f"Loaded hybrid candidates table. Total entries: {len(ym_dong_candidates)}")
print(f"Loaded target exclusions sum: {sum(target_exclusions.values()):,}건")


apt_keys: 106개


신규아파트 오더 기준 조합 개수: 260개 (월별 주소 기준)


,년도월,서비스처리센터,주소_norm,건수
0,2023-01,H072,과학성장로 33,213
1,2023-01,H072,과학성장로 77,208
2,2023-01,H072,한남로67번길 193,114
3,2023-01,H074,동서대로1754번길 60,104
4,2023-01,H072,과학성장로 80,47
...,...,...,...,...
255,2026-03,H072,대청로 53,15
256,2026-04,H075,동서대로 383,225
257,2026-04,H075,온천로 102,28
258,2026-04,H074,선화서로 85,22


## 8단계: 신규아파트 오더 배제

- 전체 오더에서 '전입신청' 및 '전입변경' 중 신규아파트(result)에 해당하는 대상을 배제합니다.
- 그 외의 소분류(전출신청, 전출변경, 연소기 연결/교체, 연소기철거)는 해당 주소/월이어도 그대로 유지합니다.

In [8]:
# 8단계: 하이브리드 관할 보정 및 신규아파트 오더 배제 (15건 미만 보호 반영)
APT_TXT = Path(r"C:\DA\apartment.txt")
apt_keys = set()
with open(APT_TXT, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        if "(" in line:
            road = line.split("(")[0].strip()
            lot = " ".join(line.split("(")[1].rstrip(")").split()[:2]).strip()
            apt_keys.add(road)
            if lot: apt_keys.add(lot)
        else:
            apt_keys.add(line)

# 주소_norm 생성 및 원래 아파트 주소 매칭 여부 원본 백업
if "주소_norm" not in df.columns:
    df["주소_norm"] = df["주소"].astype(str).str.split("(").str[0].str.strip()
df["is_apt_addr_raw"] = df["주소_norm"].apply(lambda x: any(k in x for k in apt_keys))

# 원래 데이터 백업 및 보정용 헬퍼 컬럼 생성
df["서비스처리센터_원래"] = df["서비스처리센터"].astype(str).str.strip().replace({"H051": "H073"})
df["행정동_clean"] = df["행정동"].astype(str).str.strip()

# 당월(연월) & 주소별 전입 건수 계산 (전입신청/전입변경 기준)
apt_monthly_counts = df[df["소분류"].isin(["전입신청", "전입변경"]) & df["is_apt_addr_raw"]].groupby(["년도월", "주소_norm"]).size().to_dict()

# 15건 미만 아파트 오더 보호 & 15건 이상만 제외 후보(is_apt_candidate)로 지정
def determine_apt_candidate(row):
    if not row["is_apt_addr_raw"]:
        return False
    count = apt_monthly_counts.get((row["년도월"], row["주소_norm"]), 0)
    # 15건 미만(14건 이하)이면 집계 포함 대상 (제외 후보 아님)
    if count < 15:
        return False
    # 15건 이상만 제외 후보 대상
    return True

df["is_apt_candidate"] = df.apply(determine_apt_candidate, axis=1)

# 하이브리드 관할 보정 로직 (연월별 관할 후보군 맵 기준)
def get_hybrid_corrected_bu(row):
    ym = row["년도월"]
    dong = row["행정동_clean"]
    orig_bu = row["서비스처리센터_원래"]
    
    key = (ym, dong)
    if key in ym_dong_candidates:
        candidates = ym_dong_candidates[key]
        if orig_bu in candidates:
            return orig_bu
        else:
            return max(candidates, key=candidates.get) # 실적이 가장 많은 센터로 매핑
    return orig_bu

df["서비스처리센터"] = df.apply(get_hybrid_corrected_bu, axis=1)

# 인덱스 초기화 및 백업
df = df.reset_index(drop=True)
df["order_idx"] = df.index

excluded_indices = set()
grouped = df.groupby(["서비스처리센터", "행정동_clean", "년도월"])

for (bu, dong, ym), target_n in target_exclusions.items():
    if target_n <= 0: continue
    try:
        sub_df = grouped.get_group((bu, dong, ym))
    except KeyError:
        continue
        
    # 1순위: 15건 이상인 아파트 전입 오더 중에서 제외
    candidate1 = sub_df[sub_df["소분류"].isin(["전입신청", "전입변경"]) & (sub_df["is_apt_candidate"] == True)]
    candidate1 = candidate1.sort_values(by=["마감일", "order_idx"])
    selected_idx = list(candidate1["order_idx"].values[:target_n])
    excluded_indices.update(selected_idx)
    
    n_selected = len(selected_idx)
    if n_selected < target_n:
        needed = target_n - n_selected
        # 2순위: 15건 미만인 아파트는 절대 건드리지 않고, 아파트 주소와 전혀 무관한 순수 일반 오더에서만 추가 제외
        candidate2 = sub_df[sub_df["is_apt_addr_raw"] == False]
        candidate2 = candidate2.sort_values(by=["마감일", "order_idx"])
        selected_idx2 = list(candidate2["order_idx"].values[:needed])
        excluded_indices.update(selected_idx2)

df["오더구분"] = np.where(df["order_idx"].isin(excluded_indices), "신규아파트오더", "실제오더")
df_pure = df[df["오더구분"] == "실제오더"].drop(columns=["order_idx", "is_apt_addr_raw", "is_apt_candidate", "오더구분", "서비스처리센터_원래", "행정동_clean"])

print(f"배제 전 전체 오더: {len(df):,}행")
print(f"배제 후 순수 오더: {len(df_pure):,}행")
print(f"배제된 신규아파트 오더 건수: {len(excluded_indices):,}행")


배제 전 전체 오더: 1,123,695행
배제 후 순수 오더: 1,095,591행
배제된 신규아파트 오더 건수: 28,104행


## 9단계: 마감일 기준 일자 및 연도 변환

- **마감일** 컬럼을 기준으로 정형화하여 연도(`23년`, `24년`, `25년`, `26년`)와 일자(`MM.DD`, 예: `01.01`) 컬럼을 생성합니다.
- 2024년 윤년을 포함한 366일 날짜 템플릿과 4개 연도 템play을 생성합니다.

In [9]:
# 마감일 컬럼의 앞 10자리를 자르고 온점을 대시로 변환
df_pure["마감일_clean"] = df_pure["마acao일" if "마acao일" in df_pure.columns else "마감일"].astype(str).str.slice(0, 10).str.replace(".", "-", regex=False)
df_pure["마감일_dt"] = pd.to_datetime(df_pure["마감일_clean"], errors="coerce")

# 연도 변환 및 매핑
df_pure["연도"] = df_pure["마감일_dt"].dt.year.map({
    2023: "23년",
    2024: "24년",
    2025: "25년",
    2026: "26년"
})

# 일자 변환 (MM.DD)
df_pure["일자"] = df_pure["마감일_dt"].dt.strftime("%m.%d")

# 윤년(366일) 기준 날짜 템플릿 및 연도 템플릿
all_days = pd.date_range(start="2024-01-01", end="2024-12-31").strftime("%m.%d").tolist()
all_years = ["23년", "24년", "25년", "26년"]

print("마감일 기준 날짜 및 연도 변환 완료")
print(f"마감일 파싱 실패(누락 포함) 건수: {df_pure['마감일_dt'].isna().sum():,}건")

마감일 기준 날짜 및 연도 변환 완료
마감일 파싱 실패(누락 포함) 건수: 1,859건


## 10단계: 사업부별 일별/연도별 피벗 생성 및 엑셀 저장

- 5개 사업부(H071, H072, H073, H074, H075)별로 각각 피벗 테이블을 생성합니다.
- 행에는 `01.01` ~ `12.31` (366일)이 모두 포함되게 하고, 열에는 `23년` ~ `26년` 실적이 나오게 구성합니다.
- 각 결과를 5개 시트로 구성하여 엑셀 파일로 저장합니다.

In [10]:
# 10단계: 사업부별 일별 피벗 생성 및 엑셀 저장 (가독성 리팩토링 및 덮어쓰기 안전 장치)
def generate_daily_pivot(df_source, bu_code, days_template, years_template):
    """
    특정 사업부의 일별, 연도별 오더 건수 피벗 테이블 생성
    """
    # 사업부 필터
    df_bu = df_source[df_source["서비스처리센터"] == bu_code]
    
    # 일별, 연도별 오더 건수 피벗 테이블 생성
    pivot_bu = df_bu.pivot_table(
        index="일자",
        columns="연도",
        values="오더번호",
        aggfunc="count",
        fill_value=0
    )
    
    # 366일 및 4개 연도 템플릿으로 reindex (오더가 없는 날은 0으로 채움)
    pivot_bu = pivot_bu.reindex(index=days_template, columns=years_template, fill_value=0)
    pivot_bu.index.name = "날짜"
    return pivot_bu

# 집계 및 파일 내보내기 실행
business_units = ["H071", "H072", "H073", "H074", "H075"]
output_path = Path(r"C:\DA\사업부별_일별오더집계.xlsx")
temp_output_path = Path(r"C:\DA\사업부별_일별오더집계_temp.xlsx")

print("시작: 사업부별 일별 피벗 테이블 생성 및 내보내기")
try:
    # Excel이 열려 있어 발생하는 쓰기/권한 오류 방지를 위해 임시 파일로 먼저 기록
    with pd.ExcelWriter(temp_output_path, engine="openpyxl") as writer:
        for bu in business_units:
            pivot_bu = generate_daily_pivot(df_pure, bu, all_days, all_years)
            pivot_bu.to_excel(writer, sheet_name=bu)
            print(f"[{bu}] 피벗 생성 완료 (shape: {pivot_bu.shape})")
            
    # 임시 저장이 성공하면 최종 경로 파일 덮어쓰기 (기존 파일 안전 제거 후 이동)
    if output_path.exists():
        os.remove(output_path)
    shutil.move(temp_output_path, output_path)
    print(f"\n성공: 최종 엑셀 파일 저장 완료 -> {output_path}")
    
except PermissionError:
    print(f"\n[오류] 최종 저장 파일이 사용 중이거나 열려 있습니다: {output_path}")
    print(f"-> 수집된 데이터는 임시 파일 {temp_output_path}에 임시 백업 완료되었습니다.")
except Exception as e:
    print(f"\n[오류] 피벗 데이터 처리 중 예외 발생: {e}")
    if temp_output_path.exists():
        os.remove(temp_output_path)


[H071] 피벗 생성 완료 (시트 저장 완료, shape: (366, 4))
연도     23년  24년  25년  26년
날짜                       
01.01   67   94  127  185
01.02  198  237  165  212
01.03  180  183  203  211
01.04  151  165  187  108
01.05  128  213   91  199
[H072] 피벗 생성 완료 (시트 저장 완료, shape: (366, 4))
연도     23년  24년  25년  26년
날짜                       
01.01  116   75  143  172
01.02  254  247  182  262
01.03  219  197  206  251
01.04  188  164  234  124
01.05  190  217  107  215


[H073] 피벗 생성 완료 (시트 저장 완료, shape: (366, 4))
연도     23년  24년  25년  26년
날짜                       
01.01   89   82  100  111
01.02  261  220  195  209
01.03  185  204  234  189
01.04  183  184  214   80
01.05  194  266   96  240


[H074] 피벗 생성 완료 (시트 저장 완료, shape: (366, 4))
연도     23년  24년  25년  26년
날짜                       
01.01  133  100  192  206
01.02  282  268  205  267
01.03  274  231  242  224
01.04  228  215  262  106
01.05  216  262  154  244


[H075] 피벗 생성 완료 (시트 저장 완료, shape: (366, 4))
연도     23년  24년  25년  26년
날짜                       
01.01  169  157  206  221
01.02  348  331  269  294
01.03  276  262  311  307
01.04  175  205  284  196
01.05  201  292  188  271
최종 엑셀 파일 저장 완료: C:\DA\사업부별_일별오더집계.xlsx


## 11단계: 월별/사업부별 마감일 날짜 누락(파싱 실패) 건수 분석

In [ ]:
# 날짜 누락(마감일 파싱 실패) 행 필터
df_missing = df_pure[df_pure["마감일_dt"].isna()]

# 사업부별 & 월별 누락 건수 피벗 테이블 생성
if not df_missing.empty:
    missing_summary = df_missing.groupby(["년도월", "서비스처리센터"]).size().unstack(fill_value=0)
    print("--- [사업부별 & 월별] 마감일 날짜 누락(파싱 실패) 건수 ---")
    print(missing_summary)
    print(f"\n총 누락 건수: {len(df_missing):,}건")
else:
    print("마감일 날짜 누락 건이 존재하지 않습니다.")

In [ ]:
# ── 주소(행정동) 누락 행 출력 ──────────────────────────────────────────────────
df_missing_addr = df[df["행정동"].isna() | (df["행정동"].astype(str).str.strip() == "")]

print(f"총 주소 누락 건수: {len(df_missing_addr):,}건")
print()

# 누락 건수를 사업부별로 요약
print("▶ 사업부별 누락 현황:")
print(df_missing_addr["사업부"].value_counts().to_string())
print()

# 상세 행 출력 (최대 50건)
display_cols = [c for c in ["오더번호", "사업부", "서비스처리센터", "주소", "행정동", "소분류", "마감일"] if c in df.columns]
display(df_missing_addr[display_cols].head(50))


In [ ]:
# ── 주소(행정동) 누락 행 출력 ──────────────────────────────────────────────────
df_missing_addr = df[df["행정동"].isna() | (df["행정동"].astype(str).str.strip() == "")]

print(f"총 주소 누락 건수: {len(df_missing_addr):,}건")
print()

# 누락 건수를 사업부별로 요약
print("▶ 사업부별 누락 현황:")
print(df_missing_addr["사업부"].value_counts().to_string())
print()

# 상세 행 출력 (최대 50건)
display_cols = [c for c in ["오더번호", "사업부", "서비스처리센터", "주소", "행정동", "소분류", "마감일"] if c in df.columns]
display(df_missing_addr[display_cols].head(50))


In [ ]:
# ── 신규아파트오더로 제외(배제)된 데이터 출력 ───────────────────────────────
# df_pure를 생성할 때 배제된 'excluded_indices'에 해당하는 행들을 출력합니다.
# (이전 단계에서 df['오더구분'] == '신규아파트오더'로 처리된 데이터입니다.)

if '오더구분' in df.columns:
    df_excluded = df[df["오더구분"] == "신규아파트오더"]
else:
    # 만약 df_pure가 이미 생성되어 df가 실제오더만 남은 상태라면, order_idx 백업본이나
    # 제외 인덱스(excluded_indices) 리스트가 남아있는 경우에 한해 아래처럼 복구하여 필터링합니다.
    # (노트북 흐름상 df가 갱신되기 직전 단계 기준으로 출력 가능)
    try:
        df_excluded = df_tot[df_tot["order_idx"].isin(excluded_indices)]
    except NameError:
        df_excluded = df[~df.index.isin(df_pure.index)]

print(f"★ 제외(배제) 처리된 신규아파트오더 건수: {len(df_excluded):,}건")
print()

# 제외된 건수를 사업부/행정동별로 요약
if len(df_excluded) > 0:
    print("▶ 사업부별 제외 현황:")
    print(df_excluded["서비스처리센터"].value_counts().to_string())
    print()
    
    print("▶ 행정동별 제외 TOP 10:")
    print(df_excluded["행정동"].value_counts().head(10).to_string())
    print()
    
    # 상세 제외 행 출력 (최대 50건)
    display_cols = [c for c in ["오더번호", "서비스처리센터", "행정동", "주소", "소분류", "마감일", "년도월"] if c in df_excluded.columns]
    display(df_excluded[display_cols].head(50))
else:
    print("제외된 데이터가 없습니다.")


In [ ]:
# ── 보정 대상 11건 상세 오더 출력 (H074: 7건, H075: 4건) ──────────────────────
# 이미 메모리에 로드되어 전처리가 완료된 df 변수를 그대로 활용합니다.

print("▶ [2023-11] H075 원신흥동 (보정 대상 3건)")
display(df[(df["서비스처리센터_원래"] == "H075") & (df["행정동_clean"] == "원신흥동") & (df["년도월"] == "2023-11")][["오더번호", "소분류", "마감일", "주소"]].head(3))
print("-" * 80)

print("▶ [2025-06] H074 선화동 (보정 대상 7건)")
display(df[(df["서비스처리센터_원래"] == "H074") & (df["행정동_clean"] == "선화동") & (df["년도월"] == "2025-06")][["오더번호", "소분류", "마감일", "주소"]].head(7))
print("-" * 80)

print("▶ [2023-10] H075 원신흥동 (보정 대상 1건)")
display(df[(df["서비스처리센터_원래"] == "H075") & (df["행정동_clean"] == "원신흥동") & (df["년도월"] == "2023-10")][["오더번호", "소분류", "마감일", "주소"]].head(1))
